# Non-Markovian Operator Learning and Thermodynamic Consistency in Non-Equilibrium Systems

Problem Formulation
Standard Physics-Informed Neural Networks (PINNs) and neural operators (such as DeepONet and Fourier Neural Operators) typically assume Markovian dynamics governed by local integer-order differential operators. However, complex non-equilibrium systems—such as glassy materials, active matter, and driven lattice gases—exhibit temporal memory and non-locality, where the system's history directly dictates its future evolution.

Mathematically, these dynamics are governed by fractional time derivatives (e.g., Caputo derivatives) or generalized Langevin equations containing Volterra-type integral memory kernels:

$$\frac{\partial u(x, t)}{\partial t} = \mathcal{L}[u(x, t)] + \int_{0}^{t} K(t - \tau) \mathcal{N}[u(x, \tau)] \, d\tau + \xi(x, t)$$

Research QuestionHow can we construct a physics-informed neural operator architecture that simultaneously reconstructs unknown non-local memory kernels $K(t-\tau)$ and fractional dynamics from sparse, noisy trajectory data, while strictly preserving thermodynamic consistency (such as local entropy production bounds, flux balance in non-equilibrium steady states, and generalized fluctuation-dissipation relations)?

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CompletelyMonotoneKernel(nn.Module):
    """
    Parameterizes a physically valid memory kernel K(dt) >= 0, K'(dt) <= 0
    using a Laplace transform representation (sum of decaying exponentials).
    """
    def __init__(self, num_modes: int = 16):
        super().__init__()
        # Learnable positive weights and decay rates
        self.raw_weights = nn.Parameter(torch.randn(num_modes) * 0.1)
        self.raw_rates = nn.Parameter(torch.linspace(-2.0, 2.0, num_modes))

    def forward(self, dt: torch.Tensor) -> torch.Tensor:
        """
        dt: Tensor of time lags of shape (T,)
        Returns: Kernel values K(dt) of shape (T,)
        """
        weights = F.softplus(self.raw_weights)  # Enforce w_i > 0
        rates = F.softplus(self.raw_rates)      # Enforce lambda_i > 0
        # K(dt) = sum_i w_i * exp(-lambda_i * dt)
        kernel = torch.sum(weights.unsqueeze(0) * torch.exp(-dt.unsqueeze(1) * rates.unsqueeze(0)), dim=-1)
        return kernel


class SpectralSpatialOperator(nn.Module):
    """
    1D Fourier Neural Operator (FNO) layer to capture spatial non-locality.
    """
    def __init__(self, channels: int, modes: int):
        super().__init__()
        self.channels = channels
        self.modes = modes
        scale = 1.0 / (channels * channels)
        self.weights = nn.Parameter(scale * torch.randn(channels, channels, modes, dtype=torch.cfloat))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (Batch, Channels, Spatial_Grid)
        B, C, N = x.shape
        x_ft = torch.fft.rfft(x, dim=-1)
        out_ft = torch.zeros(B, C, N // 2 + 1, dtype=torch.cfloat, device=x.device)
        
        out_ft[:, :, :self.modes] = torch.einsum(
            "bix,iox->box", x_ft[:, :, :self.modes], self.weights
        )
        return torch.fft.irfft(out_ft, n=N, dim=-1)


class NonMarkovianNeuralOperator(nn.Module):
    """
    Evolves a 1D field u(x, t) using a local Markovian operator + Volterra memory integral.
    """
    def __init__(self, grid_size: int, hidden_dim: int = 32, modes: int = 12, memory_modes: int = 16):
        super().__init__()
        self.lift = nn.Conv1d(1, hidden_dim, kernel_size=1)
        self.markov_op = SpectralSpatialOperator(hidden_dim, modes)
        self.memory_op = SpectralSpatialOperator(hidden_dim, modes)
        self.kernel = CompletelyMonotoneKernel(num_modes=memory_modes)
        self.project = nn.Conv1d(hidden_dim, 1, kernel_size=1)

    def forward(self, u_history: torch.Tensor, dt: float) -> torch.Tensor:
        """
        u_history: Past trajectory of shape (Batch, Time_Steps, Spatial_Grid)
        dt: Uniform time step size
        Returns: Predicted du/dt at the current step (Batch, Spatial_Grid)
        """
        B, T, N = u_history.shape
        device = u_history.device

        # Lift trajectory to latent feature space: (B*T, C, N)
        u_flat = u_history.reshape(B * T, 1, N)
        h_flat = F.gelu(self.lift(u_flat))

        # 1. Instantaneous Markovian contribution at t = T-1
        h_current = h_flat.reshape(B, T, -1, N)[:, -1]
        dh_markov = self.markov_op(h_current)

        # 2. Non-local Volterra memory integral over tau in [0, t]
        h_mem_all = self.memory_op(h_flat).reshape(B, T, -1, N)
        
        # Compute time lags: [T*dt, (T-1)*dt, ..., dt]
        lags = torch.arange(T, 0, -1, device=device, dtype=torch.float32) * dt
        k_vals = self.kernel(lags)  # Shape: (T,)

        # Numerical integration (Riemann sum / trapezoidal approximation)
        # Contract time dimension with memory kernel
        dh_memory = torch.einsum("btcn,t->bcn", h_mem_all, k_vals) * dt

        # Combine and project back to physical field
        dh_total = dh_markov + dh_memory
        du_dt = self.project(dh_total).squeeze(1)
        return du_dt


def thermodynamic_loss(u_pred_next: torch.Tensor, u_true_next: torch.Tensor, u_current: torch.Tensor, dt: float) -> torch.Tensor:
    """
    Combines data fidelity with a structural thermodynamic constraint
    (e.g., bounding free-energy dissipation / entropy production rate).
    """
    # 1. Data MSE loss
    mse_loss = F.mse_loss(u_pred_next, u_true_next)

    # 2. Example Physical Constraint: Discrete Free Energy Dissipation
    # For a purely relaxational system F[u] = 0.5 * integral(u^2 + |grad u|^2) dx,
    # dF/dt must be <= 0. (Modify functional F based on specific NESS driving force).
    def free_energy(u):
        grad_u = torch.diff(u, dim=-1)
        return 0.5 * (torch.mean(u**2, dim=-1) + torch.mean(grad_u**2, dim=-1))

    dF_dt = (free_energy(u_pred_next) - free_energy(u_current)) / dt
    # Penalize positive free energy production (unphysical energy creation)
    thermo_penalty = torch.mean(F.relu(dF_dt))

    return mse_loss + 0.1 * thermo_penalty


# --- Quick Execution Sanity Check ---
if __name__ == "__main__":
    batch_size, time_steps, grid_size = 8, 20, 64
    dt = 0.05

    # Synthetic noisy trajectory data
    u_hist = torch.randn(batch_size, time_steps, grid_size)
    u_next_true = torch.randn(batch_size, grid_size)

    model = NonMarkovianNeuralOperator(grid_size=grid_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    # Forward Euler integration step
    du_dt = model(u_hist, dt)
    u_next_pred = u_hist[:, -1, :] + dt * du_dt

    loss = thermodynamic_loss(u_next_pred, u_next_true, u_hist[:, -1, :], dt)
    loss.backward()
    optimizer.step()

    print(f"Step successful | Total Loss: {loss.item():.6f} | Learned Kernel[0]: {model.kernel(torch.tensor([dt])).item():.4f}")

Step successful | Total Loss: 2.126933 | Learned Kernel[0]: 10.4348
